In [1]:
import pandas as pd
import numpy as np

# 🔹 Load Excel file
file_path = "D:/Part Production Report/PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval
z = 1.96
lower_ci = mean_demand - z * std_error
upper_ci = mean_demand + z * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


              Material   Mean_Demand      Std_Dev   Lower_95_CI   Upper_95_CI
0        S01113-000E00  17153.846154  2351.104487  15875.771196  18431.921112
1        S22127-000E01   7120.615385  1940.953612   6065.500982   8175.729787
2        C51019-000E00   6861.538462  1846.410239   5857.818422   7865.258501
3        S03078-000E00   2849.538462  2072.130973   1723.115233   3975.961690
4        S31775-000E00   1053.000000  1691.449378    133.517644   1972.482356
..                 ...           ...          ...           ...           ...
281      S01095-000E00      0.000000     0.000000      0.000000      0.000000
282      S31742-000E00      0.000000     0.000000      0.000000      0.000000
283  14SW010135-0001S0      0.000000     0.000000      0.000000      0.000000
284  14SW210289-0001S0     11.538462    41.602515    -11.076923     34.153846
285      M41124-000S00      4.615385    16.641006     -4.430769     13.661538

[286 rows x 5 columns]


In [2]:
import pandas as pd
import numpy as np
from scipy import stats

# 🔹 Load Excel file
file_path = "D:/Part Production Report/PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval using t-score
alpha = 0.05
# t critical value depends on df = count - 1
t_values = stats.t.ppf(1 - alpha/2, df=count - 1)

lower_ci = mean_demand - t_values * std_error
upper_ci = mean_demand + t_values * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output_using_Tscore.xlsx", index=False)

              Material   Mean_Demand      Std_Dev   Lower_95_CI   Upper_95_CI
0        S01113-000E00  17153.846154  2351.104487  15733.087932  18574.604376
1        S22127-000E01   7120.615385  1940.953612   5947.708856   8293.521913
2        C51019-000E00   6861.538462  1846.410239   5745.763921   7977.313002
3        S03078-000E00   2849.538462  2072.130973   1597.362247   4101.714676
4        S31775-000E00   1053.000000  1691.449378     30.867370   2075.132630
..                 ...           ...          ...           ...           ...
281      S01095-000E00      0.000000     0.000000      0.000000      0.000000
282      S31742-000E00      0.000000     0.000000      0.000000      0.000000
283  14SW010135-0001S0      0.000000     0.000000      0.000000      0.000000
284  14SW210289-0001S0     11.538462    41.602515    -13.601686     36.678610
285      M41124-000S00      4.615385    16.641006     -5.440675     14.671444

[286 rows x 5 columns]


In [12]:
import pandas as pd
import re

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("D:/Part Production Report/Comaprison data.xlsx")

# ===============================
# EXTRACT ALL DATES
# ===============================

columns = df.columns.tolist()

dates = sorted(
    list(set(re.findall(r"\d{4}-\d{2}-\d{2}", " ".join(columns))))
)

# ===============================
# DAILY DEVIATION
# ===============================

for date in dates:
    
    indent_col = f"{date} Indent"
    actual_col = f"{date} Total Production Plan"
    
    if indent_col in df.columns and actual_col in df.columns:
        
        # Daily deviation
        df[f"{date} Deviation"] = df[actual_col] - df[indent_col]
        
        # Percentage deviation
        df[f"{date} Deviation_%"] = (
            df[f"{date} Deviation"] / df[indent_col]
        ) * 100

# ===============================
# MONTH TILL DATE SUMMARY
# ===============================

indent_cols = [
    f"{d} Indent"
    for d in dates
    if f"{d} Indent" in df.columns
]

actual_cols = [
    f"{d} Total Production Plan"
    for d in dates
    if f"{d} Total Production Plan" in df.columns
]

df["Total_Indent_Till_Date"] = df[indent_cols].sum(axis=1)
df["Total_Actual_Till_Date"] = df[actual_cols].sum(axis=1)

df["Total_Deviation"] = (
    df["Total_Actual_Till_Date"] - df["Total_Indent_Till_Date"]
)

df["Total_Deviation_%"] = (
    df["Total_Deviation"] / df["Total_Indent_Till_Date"]
) * 100

# ===============================
# SAVE
# ===============================

df.to_excel("indent_vs_actual_wide_analysis.xlsx", index=False)

print("Indent vs Actual comparison completed successfully.")

Indent vs Actual comparison completed successfully.


In [13]:
import pandas as pd
import numpy as np
import re
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("C:/Users\Ex0164/indent_vs_actual_wide_analysis.xlsx")

# ===============================
# IDENTIFY ACTUAL COLUMNS
# ===============================

columns = df.columns.tolist()

actual_cols = [
    col for col in columns
    if "Total Production Plan" in col
]

# ===============================
# CALCULATE μ AND σ PER MATERIAL
# ===============================

df["Mean_Actual"] = df[actual_cols].mean(axis=1)
df["Std_Actual"] = df[actual_cols].std(axis=1)

# Replace NaN std (if only 1 value)
df["Std_Actual"] = df["Std_Actual"].fillna(0)

# ===============================
# CONFIDENCE BAND
# ===============================

df["CI_Lower"] = df["Mean_Actual"] - Z * df["Std_Actual"]
df["CI_Upper"] = df["Mean_Actual"] + Z * df["Std_Actual"]

# Avoid negative lower bound
df["CI_Lower"] = df["CI_Lower"].apply(lambda x: max(0, x))

# ===============================
# SAVE OUTPUT
# ===============================

df.to_excel("confidence_band_analysis.xlsx", index=False)

print("Confidence interval (μ ± Zσ) calculated successfully.")

Confidence interval (μ ± Zσ) calculated successfully.


In [1]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

ci_df = pd.read_excel("C:/Users/Ex0164/confidence_band_analysis.xlsx")
actual_df = pd.read_excel("D:/Part Production Report/PLAN_ACTUAL.xlsx")

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB
# =====================================

df["12th_Inside_CI"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper"])
)

# =====================================
# VALIDATE 13TH FEB
# =====================================

df["13th_Inside_CI"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_CI"].mean() * 100
hit_rate_13 = df["13th_Inside_CI"].mean() * 100

print("12th Feb Hit Rate:", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate:", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("CI_validation_12_13.xlsx", index=False)

12th Feb Hit Rate: 89.86 %
13th Feb Hit Rate: 86.71 %


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("C:/Users/Ex0164/indent_vs_actual_wide_analysis.xlsx")

# ===============================
# IDENTIFY INDENT COLUMNS
# ===============================

indent_cols = [
    col for col in df.columns
    if "Indent" in col
]

# ===============================
# CALCULATE μ AND σ ON INDENT
# ===============================

df["Mean_Indent"] = df[indent_cols].mean(axis=1)
df["Std_Indent"] = df[indent_cols].std(axis=1)

df["Std_Indent"] = df["Std_Indent"].fillna(0)

# ===============================
# CONFIDENCE BAND ON INDENT
# ===============================

df["Indent_CI_Lower"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower"] = df["Indent_CI_Lower"].apply(lambda x: max(0, x))

# ===============================
# SAVE
# ===============================

df.to_excel("indent_confidence_band.xlsx", index=False)

print("Indent-based confidence interval calculated.")

Indent-based confidence interval calculated.
